# Dataset Split

This notebook handles the final dataset-splitting stage for the Cat, Dog, Bird, and Unknown image classifier. The goal of this notebook is to take the manually cleaned image pools from `Model Building/Datasets/Manually Cleaned` and create the final Train, Validation, and Test dataset structure.

Before running Notebook 2, all Test and Train/Validation data from `Model Building/Datasets/Auto Cleaned` must be manually reviewed, and all valid images must be moved into the `Model Building/Datasets/Manually Cleaned` folders.

Here is the required structure:

```text
Model Building/
└── Datasets/
    └── Manually Cleaned/
        ├── Train and Validation/
        │   ├── Cat/
        │   ├── Dog/
        │   ├── Bird/
        │   └── Unknown/
        └── Test/
            ├── Cat/
            ├── Dog/
            ├── Bird/
            └── Unknown/
```

The manually cleaned Train/Validation pools are checked against the expected class totals and then split into Train and Validation using a source-wise 90/10 split. The manually cleaned Test pool is copied into the final Test structure.

## 1. Dataset Paths and Settings

This section defines the dataset paths, class order, supported image formats, random seed, and Train/Validation split ratio.

The required `Model Building/Datasets/Manually Cleaned/Train and Validation` and `Model Building/Datasets/Manually Cleaned/Test` folders must already exist before the final dataset split begins.

In [5]:
from pathlib import Path
import random
import shutil

from tqdm.auto import tqdm


RANDOM_SEED = 42
TRAIN_RATIO = 0.90

class_names = ["Cat", "Dog", "Bird", "Unknown"]

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}

expected_train_val_counts = {
    "Cat": 30_299,
    "Dog": 30_952,
    "Bird": 30_999,
    "Unknown": 54_754,
}

expected_test_counts = {
    "Cat": 2_562,
    "Dog": 2_885,
    "Bird": 2_640,
    "Unknown": 2_682,
}


current_dir = Path.cwd().resolve()
model_building_dir = None

for directory in [current_dir, *current_dir.parents]:
    if directory.name == "Model Building":
        model_building_dir = directory
        break

    candidate_dir = directory / "Model Building"

    if candidate_dir.is_dir():
        model_building_dir = candidate_dir
        break


if model_building_dir is None:
    raise FileNotFoundError(
        "The `Model Building` directory could not be located."
    )


datasets_dir = model_building_dir / "Datasets"

manually_cleaned_train_val_dir = (
    datasets_dir
    / "Manually Cleaned"
    / "Train and Validation"
)

manually_cleaned_test_dir = (
    datasets_dir
    / "Manually Cleaned"
    / "Test"
)

final_dataset_dir = datasets_dir / "Final Dataset"
final_train_dir = final_dataset_dir / "Train"
final_validation_dir = final_dataset_dir / "Validation"
final_test_dir = final_dataset_dir / "Test"


def get_image_files(directory):
    return sorted(
        path
        for path in directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in image_extensions
    )


required_directories = {
    "Model Building/Datasets/Manually Cleaned/Train and Validation":
        manually_cleaned_train_val_dir,
    "Model Building/Datasets/Manually Cleaned/Test":
        manually_cleaned_test_dir,
}


for relative_name, directory in required_directories.items():
    if not directory.is_dir():
        raise FileNotFoundError(
            f"Required directory does not exist: `{relative_name}`"
        )


for class_name in class_names:
    train_val_class_dir = (
        manually_cleaned_train_val_dir / class_name
    )

    test_class_dir = (
        manually_cleaned_test_dir / class_name
    )

    if not train_val_class_dir.is_dir():
        raise FileNotFoundError(
            "Required directory does not exist: "
            f"`Model Building/Datasets/Manually Cleaned/"
            f"Train and Validation/{class_name}`"
        )

    if not test_class_dir.is_dir():
        raise FileNotFoundError(
            "Required directory does not exist: "
            f"`Model Building/Datasets/Manually Cleaned/"
            f"Test/{class_name}`"
        )


print("Dataset paths and settings are ready.")
print(f"Random seed:  {RANDOM_SEED}")
print("Split method: Source-wise 90/10")

Dataset paths and settings are ready.
Random seed:  42
Split method: Source-wise 90/10


## 2. Final Dataset Split and Verification

The manually cleaned Train/Validation images are checked against the expected class totals and split separately within each dataset source using a deterministic 90/10 split and random seed 42.

Splitting each source separately keeps every source represented in both Train and Validation. The copied Train, Validation, and Test images are stored directly inside their final class folders and renamed.

The original images in `Model Building/Datasets/Manually Cleaned` remain unchanged. After copying is complete, the final Train, Validation, and Test totals are checked against the manually cleaned input totals.

In [6]:
actual_train_val_counts = {
    class_name: len(
        get_image_files(
            manually_cleaned_train_val_dir / class_name
        )
    )
    for class_name in class_names
}

actual_test_counts = {
    class_name: len(
        get_image_files(
            manually_cleaned_test_dir / class_name
        )
    )
    for class_name in class_names
}


count_errors = []

for class_name in class_names:
    actual_train_val_count = actual_train_val_counts[class_name]
    expected_train_val_count = expected_train_val_counts[class_name]

    actual_test_count = actual_test_counts[class_name]
    expected_test_count = expected_test_counts[class_name]

    if actual_train_val_count != expected_train_val_count:
        count_errors.append(
            f"Train/Validation {class_name}: expected "
            f"{expected_train_val_count:,}, found "
            f"{actual_train_val_count:,}"
        )

    if actual_test_count != expected_test_count:
        count_errors.append(
            f"Test {class_name}: expected "
            f"{expected_test_count:,}, found "
            f"{actual_test_count:,}"
        )


if count_errors:
    raise ValueError(
        "The manually cleaned image counts do not match:\n\n"
        + "\n".join(count_errors)
    )


total_input_train_val = sum(
    actual_train_val_counts.values()
)

total_input_test = sum(
    actual_test_counts.values()
)

total_images_to_copy = (
    total_input_train_val
    + total_input_test
)


print("Manually Cleaned counts verified.")
print(f"Train/Validation images: {total_input_train_val:,}")
print(f"Test images:             {total_input_test:,}")
print()


if final_dataset_dir.exists():
    shutil.rmtree(final_dataset_dir)


for split_dir in [
    final_train_dir,
    final_validation_dir,
    final_test_dir,
]:
    for class_name in class_names:
        (split_dir / class_name).mkdir(
            parents=True,
            exist_ok=True,
        )


with tqdm(
    total=total_images_to_copy,
    desc="Creating Final Dataset",
    unit="image",
) as progress:

    for class_name in class_names:
        class_dir = (
            manually_cleaned_train_val_dir / class_name
        )

        source_groups = {}

        for image_path in get_image_files(class_dir):
            relative_path = image_path.relative_to(
                class_dir
            )

            source_name = (
                relative_path.parts[0]
                if len(relative_path.parts) > 1
                else "Class Root"
            )

            source_groups.setdefault(
                source_name,
                [],
            ).append(image_path)


        class_train_files = []
        class_validation_files = []

        for source_name, source_files in sorted(
            source_groups.items()
        ):
            source_files = sorted(source_files)

            if len(source_files) < 2:
                raise ValueError(
                    f"{class_name}/{source_name} contains fewer "
                    "than two images and cannot be split."
                )

            random_generator = random.Random(
                f"{RANDOM_SEED}:{class_name}:{source_name}"
            )

            random_generator.shuffle(source_files)

            validation_count = len(source_files) // 10

            validation_count = max(
                1,
                min(
                    validation_count,
                    len(source_files) - 1,
                ),
            )

            class_validation_files.extend(
                source_files[:validation_count]
            )

            class_train_files.extend(
                source_files[validation_count:]
            )


        for image_number, image_path in enumerate(
            class_train_files,
            start=1,
        ):
            destination_name = (
                f"train_{image_number:06d}"
                f"{image_path.suffix.lower()}"
            )

            destination_path = (
                final_train_dir
                / class_name
                / destination_name
            )

            shutil.copy2(
                image_path,
                destination_path,
            )

            progress.update(1)


        for image_number, image_path in enumerate(
            class_validation_files,
            start=1,
        ):
            destination_name = (
                f"validation_{image_number:06d}"
                f"{image_path.suffix.lower()}"
            )

            destination_path = (
                final_validation_dir
                / class_name
                / destination_name
            )

            shutil.copy2(
                image_path,
                destination_path,
            )

            progress.update(1)


    for class_name in class_names:
        class_dir = (
            manually_cleaned_test_dir / class_name
        )

        test_files = get_image_files(class_dir)

        for image_number, image_path in enumerate(
            test_files,
            start=1,
        ):
            destination_name = (
                f"test_{image_number:06d}"
                f"{image_path.suffix.lower()}"
            )

            destination_path = (
                final_test_dir
                / class_name
                / destination_name
            )

            shutil.copy2(
                image_path,
                destination_path,
            )

            progress.update(1)


final_counts = {}

for class_name in class_names:
    final_counts[class_name] = {
        "Train": len(
            get_image_files(
                final_train_dir / class_name
            )
        ),
        "Validation": len(
            get_image_files(
                final_validation_dir / class_name
            )
        ),
        "Test": len(
            get_image_files(
                final_test_dir / class_name
            )
        ),
    }


verification_errors = []

for class_name in class_names:
    train_count = final_counts[class_name]["Train"]
    validation_count = final_counts[class_name]["Validation"]
    test_count = final_counts[class_name]["Test"]

    final_train_val_count = (
        train_count
        + validation_count
    )

    if (
        final_train_val_count
        != actual_train_val_counts[class_name]
    ):
        verification_errors.append(
            f"{class_name} Train/Validation: expected "
            f"{actual_train_val_counts[class_name]:,}, found "
            f"{final_train_val_count:,}"
        )

    if test_count != actual_test_counts[class_name]:
        verification_errors.append(
            f"{class_name} Test: expected "
            f"{actual_test_counts[class_name]:,}, found "
            f"{test_count:,}"
        )


if verification_errors:
    raise ValueError(
        "Final Dataset verification failed:\n\n"
        + "\n".join(verification_errors)
    )


print()
print("Final Dataset counts")
print()

print(
    f"{'Class':<10}"
    f"{'Train':>12}"
    f"{'Validation':>14}"
    f"{'Test':>12}"
    f"{'Total':>12}"
)

print("-" * 60)


for class_name in class_names:
    train_count = final_counts[class_name]["Train"]
    validation_count = final_counts[class_name]["Validation"]
    test_count = final_counts[class_name]["Test"]

    class_total = (
        train_count
        + validation_count
        + test_count
    )

    print(
        f"{class_name:<10}"
        f"{train_count:>12,}"
        f"{validation_count:>14,}"
        f"{test_count:>12,}"
        f"{class_total:>12,}"
    )


print("-" * 60)


total_train = sum(
    final_counts[class_name]["Train"]
    for class_name in class_names
)

total_validation = sum(
    final_counts[class_name]["Validation"]
    for class_name in class_names
)

total_test = sum(
    final_counts[class_name]["Test"]
    for class_name in class_names
)

total_dataset = (
    total_train
    + total_validation
    + total_test
)


print(
    f"{'Total':<10}"
    f"{total_train:>12,}"
    f"{total_validation:>14,}"
    f"{total_test:>12,}"
    f"{total_dataset:>12,}"
)

print()
print("Final Dataset created and verified successfully.")
print(f"Train images:      {total_train:,}")
print(f"Validation images: {total_validation:,}")
print(f"Test images:       {total_test:,}")
print(f"Dataset total:     {total_dataset:,}")

Manually Cleaned counts verified.
Train/Validation images: 147,004
Test images:             10,769



Creating Final Dataset: 100%|██████████| 157773/157773 [04:10<00:00, 629.87image/s] 



Final Dataset counts

Class            Train    Validation        Test       Total
------------------------------------------------------------
Cat             27,272         3,027       2,562      32,861
Dog             27,858         3,094       2,885      33,837
Bird            27,901         3,098       2,640      33,639
Unknown         49,281         5,473       2,682      57,436
------------------------------------------------------------
Total          132,312        14,692      10,769     157,773

Final Dataset created and verified successfully.
Train images:      132,312
Validation images: 14,692
Test images:       10,769
Dataset total:     157,773
